# AnimalCLAP — Zero-Shot Species Classification (Colab)

This notebook loads the official **AnimalCLAP** checkpoint (`risashinoda/animalclap`) and runs zero-shot species classification on an audio file you upload.

It reuses the exact model architecture (`HFCLAPContrastive`) from the official repo [dahlian00/AnimalCLAP](https://github.com/dahlian00/AnimalCLAP), so the checkpoint loads correctly.

**Tip:** Runtime → Change runtime type → GPU (not required, but faster).

**Note:** AnimalCLAP works like CLAP — it scores your audio against a *list of candidate species names* and ranks them. It doesn't invent a species name from nothing. By default this notebook downloads the full ~6,800-species candidate list used in the paper, so in practice it behaves like open-ended classification.

In [ ]:
# 1. Install dependencies
!pip install -q torch torchaudio transformers librosa soundfile huggingface_hub pandas

In [ ]:
# 2. Download the AnimalCLAP checkpoint and the species candidate list
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="risashinoda/animalclap",
    filename="animalclap_epoch020.pth",
)

traits_csv_path = hf_hub_download(
    repo_id="risashinoda/animalclap-dataset",
    filename="species_traits.csv",
    repo_type="dataset",
)

print("Checkpoint:", ckpt_path)
print("Species list:", traits_csv_path)

In [ ]:
# 3. Model definition — copied verbatim from the official inference.py
#    (github.com/dahlian00/AnimalCLAP) so checkpoint keys line up.
import numpy as np
import torch
import torch.nn as nn
from transformers import ClapProcessor, ClapModel


class ProjectionMLP(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=512, out_dim=512):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x): return self.net(x)


class HFCLAPContrastive(nn.Module):
    def __init__(self, model_id="laion/clap-htsat-unfused",
                 proj_hidden_dim=512, proj_out_dim=512):
        super().__init__()
        self.processor = ClapProcessor.from_pretrained(model_id)
        self.backbone  = ClapModel.from_pretrained(model_id, use_safetensors=True)
        self.logit_scale = nn.Parameter(torch.tensor(np.log(1/0.07), dtype=torch.float32))
        feat_dim = getattr(getattr(self.backbone, "config", object()), "projection_dim", 512)
        self.audio_head = ProjectionMLP(feat_dim, proj_hidden_dim, proj_out_dim)
        self.text_head  = ProjectionMLP(feat_dim, proj_hidden_dim, proj_out_dim)
        self.processor.feature_extractor.do_resample = False
        self.processor.feature_extractor.return_attention_mask = False

    def _dev(self): return next(self.parameters()).device

    @staticmethod
    def _as_tensor(output):
        # Newer `transformers` versions return a BaseModelOutputWithPooling
        # from get_text_features/get_audio_features instead of a raw tensor.
        # Handle both so this works across versions.
        if torch.is_tensor(output):
            return output
        if hasattr(output, "pooler_output"):
            return output.pooler_output
        if isinstance(output, (tuple, list)):
            return output[0]
        raise TypeError(f"Unexpected feature output type: {type(output)}")

    def encode_audio(self, audio, sample_rate=48000):
        audio_list = [a.cpu().numpy() for a in audio]
        try:
            # Newer transformers: kwarg renamed 'audios' -> 'audio'
            inputs = self.processor(audio=audio_list, sampling_rate=sample_rate,
                                    return_tensors="pt", padding=True)
        except (TypeError, ValueError):
            inputs = self.processor(audios=audio_list, sampling_rate=sample_rate,
                                    return_tensors="pt", padding=True)
        inputs = {k: v.to(self._dev()) for k, v in inputs.items()}
        return self._as_tensor(self.backbone.get_audio_features(**inputs))

    def encode_text(self, texts):
        inputs = self.processor(text=texts, return_tensors="pt", padding=True)
        inputs = {k: v.to(self._dev()) for k, v in inputs.items()}
        return self._as_tensor(self.backbone.get_text_features(**inputs))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HFCLAPContrastive().to(device)

sd = torch.load(ckpt_path, map_location="cpu")
if isinstance(sd, dict) and "state_dict" in sd:
    sd = sd["state_dict"]
sd = {k.replace("module.", ""): v for k, v in sd.items()}
missing, unexpected = model.load_state_dict(sd, strict=False)
model.eval()

print(f"Loaded on {device}. Missing keys: {len(missing)}, unexpected keys: {len(unexpected)}")

In [ ]:
# 4. Audio loading — same 48kHz / 10s convention the model was trained with
import torchaudio
from torchaudio.functional import resample
import librosa

SAMPLE_RATE = 48000
CLIP_LEN = 10.0
N_SAMPLES = int(SAMPLE_RATE * CLIP_LEN)

def load_audio(path: str) -> torch.Tensor:
    try:
        wf, sr = torchaudio.load(path)
        if wf.dim() == 2:
            wf = wf.mean(dim=0)
    except Exception:
        y, sr = librosa.load(path, sr=None, mono=True)
        wf = torch.from_numpy(y)

    wf = wf.to(torch.float32)
    sr = int(sr) if sr and int(sr) > 0 else SAMPLE_RATE
    if sr != SAMPLE_RATE and wf.numel() > 1:
        wf = resample(wf, sr, SAMPLE_RATE)

    if wf.numel() >= N_SAMPLES:
        return wf[:N_SAMPLES]
    return torch.nn.functional.pad(wf, (0, N_SAMPLES - wf.numel()))

In [ ]:
# 5. Upload your audio file
from google.colab import files
uploaded = files.upload()
audio_path = list(uploaded.keys())[0]
print("Using:", audio_path)

In [ ]:
# 6. Build the candidate species list (common names) and run zero-shot classification
import pandas as pd

traits_df = pd.read_csv(traits_csv_path)

# Try to find a name column robustly. species_traits.csv only ships
# scientific_name (no common_name), so that's the realistic default.
name_col = None
for c in ["common_name", "common", "name", "scientific_name"]:
    if c in traits_df.columns:
        name_col = c
        break
if name_col is None:
    raise ValueError(f"Couldn't find a name column. Columns found: {list(traits_df.columns)}")
print(f"Using '{name_col}' as the candidate species text.")

class_texts = sorted(set(traits_df[name_col].dropna().astype(str).str.strip()))
class_texts = [c for c in class_texts if c]
print(f"Candidate species: {len(class_texts)}")

TOP_K = 5

with torch.no_grad():
    # Encode all candidate species names (batched to avoid OOM on CPU)
    text_feats = []
    B = 256
    for i in range(0, len(class_texts), B):
        batch = class_texts[i:i+B]
        feat = model.encode_text(batch)
        text_feats.append(model.text_head(feat))
    text_proj = torch.cat(text_feats, dim=0)
    text_proj = nn.functional.normalize(text_proj, dim=-1)

    # Encode the uploaded audio
    wf = load_audio(audio_path).unsqueeze(0).to(device)  # batch of 1
    a_feat = model.encode_audio(wf, sample_rate=SAMPLE_RATE)
    a_proj = model.audio_head(a_feat)
    a_proj = nn.functional.normalize(a_proj, dim=-1)

    logits = (a_proj @ text_proj.t()).squeeze(0)
    scaled_logits = logits * model.logit_scale.exp().clamp(max=100)

    # Softmax: relative ranking across all candidates (sums to ~100% across ALL classes,
    # not just the printed top-K -- that's why the printed top-5 look like they add to 100%).
    softmax_probs = torch.softmax(scaled_logits, dim=0)

    # Sigmoid: independent per-class confidence. Each class scored on its own,
    # not competing against the others, so these do NOT need to sum to 100%.
    # Not a calibrated probability (CLAP wasn't trained with a sigmoid/BCE loss),
    # but useful when you want each candidate's own score.
    sigmoid_scores = torch.sigmoid(scaled_logits)

    topk_probs, topk_idx = torch.topk(softmax_probs, k=min(TOP_K, len(class_texts)))

print(f"\nTop-{TOP_K} predictions for {audio_path}:")
print(f"  {'species':<35s}  {'softmax (relative)':>19s}  {'sigmoid (independent)':>22s}")
for rank, (p, idx) in enumerate(zip(topk_probs.tolist(), topk_idx.tolist()), start=1):
    sig = sigmoid_scores[idx].item()
    print(f"  {rank}. {class_texts[idx]:<32s}  {p*100:15.2f}%  {sig*100:18.2f}%")

### Notes
- Model, checkpoint, and architecture verified against the official repo: [github.com/dahlian00/AnimalCLAP](https://github.com/dahlian00/AnimalCLAP) and [huggingface.co/risashinoda/animalclap](https://huggingface.co/risashinoda/animalclap).
- The backbone (`laion/clap-htsat-unfused`) downloads from Hugging Face on first run (a few GB) — this is separate from the small 619MB AnimalCLAP checkpoint, which only holds the projection heads and fine-tuned weights on top of it.
- Best results come from clear ~10 second clips containing mostly the target animal's vocalization.
- `species_traits.csv` only ships scientific (Latin binomial) names, not common names, so predictions will show e.g. `Turdus migratorius` rather than `American Robin`. The paper's model was trained with scientific names as one of its valid prompt types, so this still works — you can look up the common name afterward if needed.
- If you want fewer, more relevant candidates (e.g. only local birds), filter `class_texts` before the classification cell instead of using all ~6,800 species.
- Trait inference (diet, habitat, etc.) uses a *separate* classifier head per trait that the authors trained on top of this encoder — the public repo doesn't ship pretrained trait-head weights, so that part isn't included here.